In [0]:
# 01_bronze_to_silver
storage_account_name = dbutils.secrets.get(scope="crypto-pipeline", key="storage-account-name")
storage_account_key  = dbutils.secrets.get(scope="crypto-pipeline", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("✅ Connected to Azure Data Lake")

In [0]:
# Read raw JSON from Bronze layer

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/crypto/"

df_raw = spark.read.option("multiline", "true").json(bronze_path)

print(f"✅ Loaded {df_raw.count()} records")
df_raw.printSchema()

In [0]:
display(df_raw)

In [0]:
from pyspark.sql.functions import col, to_timestamp, round

# Cast columns to correct types and clean data
df_silver = df_raw \
    .withColumn("current_price",              col("current_price").cast("double")) \
    .withColumn("market_cap",                 col("market_cap").cast("long")) \
    .withColumn("market_cap_rank",            col("market_cap_rank").cast("integer")) \
    .withColumn("total_volume",               col("total_volume").cast("long")) \
    .withColumn("high_24h",                   col("high_24h").cast("double")) \
    .withColumn("low_24h",                    col("low_24h").cast("double")) \
    .withColumn("price_change_24h",           col("price_change_24h").cast("double")) \
    .withColumn("price_change_percentage_1h", col("price_change_percentage_1h").cast("double")) \
    .withColumn("price_change_percentage_24h",col("price_change_percentage_24h").cast("double")) \
    .withColumn("price_change_percentage_7d", col("price_change_percentage_7d").cast("double")) \
    .withColumn("circulating_supply",         col("circulating_supply").cast("double")) \
    .withColumn("ath",                        col("ath").cast("double")) \
    .withColumn("ath_change_percentage",      col("ath_change_percentage").cast("double")) \
    .withColumn("fetched_at",                 to_timestamp(col("fetched_at"))) \
    .dropDuplicates(["id", "fetched_at"]) \
    .filter(col("current_price").isNotNull()) \
    .filter(col("market_cap").isNotNull())

print(f"✅ Silver records: {df_silver.count()}")
df_silver.printSchema()

In [0]:
display(df_silver)

In [0]:
# Write to Silver layer as Delta table

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/crypto/"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("market_cap_rank") \
    .save(silver_path)

print(f"✅ Written to Silver layer: {silver_path}")